In [18]:
import pandas as pd
import numpy as np
import csv
import os

In [19]:
file_path = "Airplane_Crashes_and_Fatalities_Since_1908.csv"

rows = []

with open(file_path, "r", encoding="utf-8-sig", newline="") as file:
    outer_reader = csv.reader(file)

    for record in outer_reader:
        if not record:
            continue

        if len(record) == 1:
            actual_row = next(csv.reader([record[0]]))
        else:
            actual_row = record

        rows.append(actual_row)

columns = rows[0]

bad_rows = [
    (i + 1, len(row))
    for i, row in enumerate(rows[1:])
    if len(row) != len(columns)
]

print("Number of columns:", len(columns))
print("Incorrect rows:", len(bad_rows))

if bad_rows:
    raise ValueError("Incorrect row lengths detected.")

df = pd.DataFrame(rows[1:], columns=columns)

print("\nDataset Shape:", df.shape)

display(df.head())

Number of columns: 13
Incorrect rows: 0

Dataset Shape: (5268, 13)


,Date,Time,Location,Operator,Flight #,Route,Type,Registration,cn/In,Aboard,Fatalities,Ground,Summary
0,09/17/1908,17:18,"Fort Myer, Virginia",Military - U.S. Army,,Demonstration,Wright Flyer III,,1,2,1,0,"During a demonstration flight, a U.S. Army fly..."
1,07-12-1912,06:30,"AtlantiCity, New Jersey",Military - U.S. Navy,,Test flight,Dirigible,,,5,5,0,First U.S. dirigible Akron exploded just offsh...
2,08-06-1913,,"Victoria, British Columbia, Canada",Private,-,,Curtiss seaplane,,,1,1,0,The first fatal airplane accident in Canada oc...
3,09-09-1913,18:30,Over the North Sea,Military - German Navy,,,Zeppelin L-1 (airship),,,20,14,0,The airship flew into a thunderstorm and encou...
4,10/17/1913,10:30,"Near Johannisthal, Germany",Military - German Navy,,,Zeppelin L-2 (airship),,,30,30,0,Hydrogen gas which was being vented was sucked...


In [20]:
# Clean column names
df.columns = df.columns.str.strip()

# Remove extra spaces
for col in df.columns:
    df[col] = df[col].str.strip()

# Replace empty strings with missing values
df = df.replace(r'^\s*$', pd.NA, regex=True)

# Convert numerical columns
numeric_columns = ["Aboard", "Fatalities", "Ground"]

for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Display missing values
print("Missing Values:")
print(df.isnull().sum())

# Check duplicates
print("\nDuplicate Records:", df.duplicated().sum())

print("\nFinal Dataset Shape:", df.shape)

Missing Values:
Date               0
Time            2219
Location          20
Operator          18
Flight #        4199
Route           1706
Type              27
Registration     335
cn/In           1228
Aboard            22
Fatalities        12
Ground            22
Summary          390
dtype: int64

Duplicate Records: 0

Final Dataset Shape: (5268, 13)


In [21]:
%pip install scikit-learn pyarrow huggingface_hub

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [22]:

from sklearn.model_selection import train_test_split

# First split: 70% training and 30% temporary
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=42,
    shuffle=True
)

# Second split: 15% validation and 15% testing
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    shuffle=True
)

# Display the dataset sizes
print("Training Dataset:", train_df.shape)
print("Validation Dataset:", val_df.shape)
print("Testing Dataset:", test_df.shape)

print("\nTotal Records:",
      len(train_df) + len(val_df) + len(test_df))


Training Dataset: (3687, 13)
Validation Dataset: (790, 13)
Testing Dataset: (791, 13)

Total Records: 5268


In [23]:

# Verify that there is no overlap

print("Train-Validation Overlap:",
      len(set(train_df.index) & set(val_df.index)))

print("Train-Test Overlap:",
      len(set(train_df.index) & set(test_df.index)))

print("Validation-Test Overlap:",
      len(set(val_df.index) & set(test_df.index)))

print("\nTotal Unique Records:",
      len(set(train_df.index) | set(val_df.index) | set(test_df.index)))


Train-Validation Overlap: 0
Train-Test Overlap: 0
Validation-Test Overlap: 0

Total Unique Records: 5268


In [24]:

# Save datasets in Parquet format

train_df.to_parquet("train.parquet", index=False)

val_df.to_parquet("validation.parquet", index=False)

test_df.to_parquet("test.parquet", index=False)

print("All three Parquet files created successfully!")


All three Parquet files created successfully!


In [25]:

import os

files = [
    "train.parquet",
    "validation.parquet",
    "test.parquet"
]

for file in files:
    print(file, "Exists:", os.path.exists(file))

    if os.path.exists(file):
        print("File Size:", round(os.path.getsize(file) / 1024, 2), "KB")


train.parquet Exists: True
File Size: 621.77 KB
validation.parquet Exists: True
File Size: 147.36 KB
test.parquet Exists: True
File Size: 151.05 KB


In [26]:

import pandas as pd
import os

files = {
    "Training": "train.parquet",
    "Validation": "validation.parquet",
    "Testing": "test.parquet"
}

for name, file in files.items():
    data = pd.read_parquet(file)

    print(name)
    print("Shape:", data.shape)
    print("File exists:", os.path.exists(file))
    print("-------------------------")


Training
Shape: (3687, 13)
File exists: True
-------------------------
Validation
Shape: (790, 13)
File exists: True
-------------------------
Testing
Shape: (791, 13)
File exists: True
-------------------------


In [27]:
from huggingface_hub import login

login()

In [28]:

from huggingface_hub import HfApi

api = HfApi()

repo_id = "lokesh23112007/Airplane_crashes_dataset"

files = [
    "train.parquet",
    "validation.parquet",
    "test.parquet"
]

for file in files:
    api.upload_file(
        path_or_fileobj=file,
        path_in_repo=file,
        repo_id=repo_id,
        repo_type="dataset"
    )

print("All three Parquet files uploaded successfully!")


C:\Users\Lokesh\OneDrive\Documents\dt\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (0 / 1):  83%|████████▎ |  526kB /  637kB, 48.7kB/s  
Processing Files (0 / 1): 100%|█████████▉|  637kB /  637kB, 57.4kB/s  
Processing Files (1 / 1): 100%|██████████|  637kB /  637kB, 48.4kB/s  
New Data Upload: 100%|██████████|  637kB /  637kB, 48.4kB/s  

Validating:           |   0%            

Validating:           |   0%            

Validating:           |   0%            

Validating:           |   0%            

Validating:           |   0%            

Validating: ██████████| 100%            

Validating: ██████████| 100%            

Validating: ██████████| 100%            

Processing Files (1 / 1): 100%|██████████|  637kB /  63

All three Parquet files uploaded successfully!


In [29]:
from huggingface_hub import list_repo_files

repo_files = list_repo_files(
    repo_id=repo_id,
    repo_type="dataset"
)

print("Files in Hugging Face repository:")

for file in repo_files:
    print(file)

Files in Hugging Face repository:
.gitattributes
test.parquet
train.parquet
validation.parquet


In [30]:

from huggingface_hub import HfApi, hf_hub_download
import pandas as pd

api = HfApi()

repo_id = "lokesh23112007/Airplane_crashes_dataset"

# Check repository files
repo_files = api.list_repo_files(
    repo_id=repo_id,
    repo_type="dataset"
)

required_files = [
    "train.parquet",
    "validation.parquet",
    "test.parquet"
]

print("Checking uploaded files:\n")

for file in required_files:
    if file in repo_files:
        print(file, "- FOUND")
    else:
        print(file, "- NOT FOUND")

# Download and verify each Parquet file
print("\nChecking Parquet contents:\n")

for file in required_files:
    file_path = hf_hub_download(
        repo_id=repo_id,
        filename=file,
        repo_type="dataset"
    )

    data = pd.read_parquet(file_path)

    print(file)
    print("Shape:", data.shape)
    print("-----------------------")


Checking uploaded files:

train.parquet - FOUND
validation.parquet - FOUND
test.parquet - FOUND

Checking Parquet contents:



C:\Users\Lokesh\OneDrive\Documents\dt\venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Lokesh\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
C:\Users\Lokesh\OneDrive\Documents\dt\venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cach

train.parquet
Shape: (3687, 13)
-----------------------
validation.parquet
Shape: (790, 13)
-----------------------
test.parquet
Shape: (791, 13)
-----------------------


In [31]:

# Step 16: Create the Hugging Face Dataset Card

readme_content = """---
pretty_name: Airplane Crashes and Fatalities Since 1908
configs:
- config_name: default
  data_files:
  - split: train
    path: train.parquet
  - split: validation
    path: validation.parquet
  - split: test
    path: test.parquet
---

# Airplane Crashes and Fatalities Since 1908

## Dataset Description

This dataset contains historical information about airplane crashes
and aviation accidents recorded since 1908.

It includes information about accident dates, locations, operators,
aircraft details, people aboard, fatalities, ground fatalities,
and accident summaries.

## Dataset Structure

The dataset contains 13 columns:

| Column | Description |
|---|---|
| Date | Date of the accident |
| Time | Time of the accident |
| Location | Accident location |
| Operator | Airline or aircraft operator |
| Flight # | Flight number |
| Route | Flight route |
| Type | Aircraft type |
| Registration | Aircraft registration |
| cn/In | Aircraft construction number or identification |
| Aboard | Total people aboard |
| Fatalities | Number of fatalities aboard |
| Ground | Ground fatalities |
| Summary | Accident description |

## Dataset Splits

The dataset is divided into three parts:

| Split | Records | Percentage |
|---|---:|---:|
| Training | 3,687 | 70% |
| Validation | 790 | 15% |
| Testing | 791 | 15% |
| Total | 5,268 | 100% |

## Data Preparation

- The original CSV file was loaded and inspected.
- Empty values were identified as missing values.
- Numerical columns were converted to numerical data types.
- No duplicate records were found.
- The dataset was randomly divided into three parts.
- Random state 42 was used for reproducibility.
- All three datasets were saved in Parquet format.

## File Format

All three dataset files are stored in Parquet format.

## Intended Use

This dataset is prepared for future data analysis
and machine learning experiments.

The prediction target has not yet been selected.

## Limitations

Some columns contain missing values.
The dataset requires further preprocessing depending
on the machine learning task.

## Source Information

The original CSV file was supplied for this project.
The original external source and dataset license
have not yet been independently verified.
"""

with open("README.md", "w", encoding="utf-8") as file:
    file.write(readme_content)

print("Dataset Card created successfully!")

# Upload Dataset Card to Hugging Face

from huggingface_hub import HfApi

api = HfApi()

api.upload_file(
    path_or_fileobj="README.md",
    path_in_repo="README.md",
    repo_id="lokesh23112007/Airplane_crashes_dataset",
    repo_type="dataset"
)

print("README.md uploaded successfully!")


Dataset Card created successfully!
README.md uploaded successfully!


In [32]:

import requests

repo_id = "lokesh23112007/Airplane_crashes_dataset"

# Check whether Hugging Face can process the dataset
url = "https://datasets-server.huggingface.co/is-valid"

response = requests.get(
    url,
    params={"dataset": repo_id}
)

print("HTTP Status:", response.status_code)
print("\nDataset Processing Status:")
print(response.json())

# Check detected splits
split_url = "https://datasets-server.huggingface.co/splits"

split_response = requests.get(
    split_url,
    params={"dataset": repo_id}
)

print("\nDetected Dataset Splits:")
print(split_response.json())


HTTP Status: 500

Dataset Processing Status:
{'error': 'The server is busier than usual and the response is not ready yet. Please retry later.'}

Detected Dataset Splits:
{'error': 'The server is busier than usual and the response is not ready yet. Please retry later.'}


In [33]:

from huggingface_hub import hf_hub_download
import pandas as pd

repo_id = "lokesh23112007/Airplane_crashes_dataset"

files = [
    "train.parquet",
    "validation.parquet",
    "test.parquet"
]

for file in files:

    print("\nChecking:", file)

    path = hf_hub_download(
        repo_id=repo_id,
        filename=file,
        repo_type="dataset"
    )

    data = pd.read_parquet(path)

    print("Successfully downloaded and read!")
    print("Shape:", data.shape)
    print("Columns:", len(data.columns))

print("\nAll Parquet files verified!")



Checking: train.parquet
Successfully downloaded and read!
Shape: (3687, 13)
Columns: 13

Checking: validation.parquet
Successfully downloaded and read!
Shape: (790, 13)
Columns: 13

Checking: test.parquet
Successfully downloaded and read!
Shape: (791, 13)
Columns: 13

All Parquet files verified!
